In [1]:
# ======================================================================
# SESSION 1 — Pilot: raw OHLCV + our own split/bonus adjustment
# Project: moonshot_compounding
# Scope: 6 tickers, 2015-2020, PRICE RETURN only (no dividend/total-return
#        adjustment — see chat discussion on why that's out of scope today)
# ======================================================================

# ----------------------------------------------------------------------
# STEP 0: Mount Drive and set up the folder structure we agreed on.
# We create every folder now, even ones we won't populate this session,
# so future sessions can just write into an already-existing structure
# instead of each session reinventing where things go.
# ----------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'

folders = [
    'raw/universe',
    'raw/prices',
    'corporate_actions',
    'adjusted',
    'universe',
    'features',
    'validation',
]
for f in folders:
    os.makedirs(BASE + f, exist_ok=True)

print("Folder structure created under:", BASE)
for f in folders:
    print(" -", BASE + f)


# ----------------------------------------------------------------------
# STEP 1: Install/import yfinance and pull RAW (unadjusted) OHLCV.
#
# WHY auto_adjust=False:
# yfinance's default (auto_adjust=True) silently applies its own
# split+dividend adjustment before we ever see the data — a black box
# we specifically decided not to trust blindly. Setting it to False
# gives us the untouched raw prices, so WE control what adjustment
# happens and can show our work.
# ----------------------------------------------------------------------
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# Pilot ticker list — same 6 names we picked earlier from the Bloomberg
# membership file, so we can later cross-reference this pilot against
# that file's ISIN/exch_symbol if needed.
pilot_tickers = ['RELIANCE.NS', 'HDFCBANK.NS', 'INFY.NS', 'TCS.NS', 'ITC.NS', 'WIPRO.NS']
# Note: HDFC.NS (the standalone housing finance co, Bloomberg's "HDFC IS Equity")
# merged into HDFC Bank in 2023. For this yfinance pilot we're using HDFCBANK.NS
# instead, since HDFC.NS may not have continuous data through today.
# THIS IS A GOOD EXAMPLE of exactly the ticker-identity problem we've been
# worried about — flagging it here rather than silently picking one.

START = '2015-01-01'
END = '2020-12-31'

raw_frames = []

for ticker in pilot_tickers:
    print(f"Pulling raw OHLCV for {ticker}...")

    # yf.download returns a DataFrame indexed by date, columns = OHLCV.
    # auto_adjust=False is the critical flag: gives us RAW close, not
    # yfinance's pre-adjusted close.
    df = yf.download(
        ticker,
        start=START,
        end=END,
        auto_adjust=False,   # <-- deliberate: we adjust ourselves, later
        progress=False,
    )

    if df.empty:
        # NO SILENT FAILURES: if a ticker returns nothing, we need to know
        # immediately, not discover a gap three steps downstream.
        print(f"  [WARNING] No data returned for {ticker} — investigate before proceeding.")
        continue

    # FIX (found in first run): recent yfinance versions return a MultiIndex
    # column structure like ('Close', 'RELIANCE.NS') even for a SINGLE ticker
    # download. If we don't flatten this now, every ticker ends up with its
    # own uniquely-named columns, and stacking tickers together later produces
    # a sparse, misaligned wide table instead of a proper long panel.
    # Since we're downloading one ticker at a time here, the ticker level of
    # the MultiIndex is redundant information (we already know which ticker
    # this is) — safe to drop it and keep just the field name.
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df['ticker'] = ticker
    df = df.reset_index()  # turn the Date index into a normal column
    raw_frames.append(df)


# Stack all 6 tickers into one long DataFrame: one row per (date, ticker)
raw_panel = pd.concat(raw_frames, ignore_index=True)

print(f"\nRaw pull complete. Shape: {raw_panel.shape}")
print(raw_panel.head())
print(f"\nColumns: {list(raw_panel.columns)}")

# Save this raw pull, TIMESTAMPED, so a future re-pull never silently
# overwrites what we validated today.
today_str = datetime.now().strftime('%Y-%m-%d')
raw_path = BASE + f'raw/prices/yfinance_raw_pilot_v1_{today_str}.parquet'
raw_panel.to_parquet(raw_path, index=False)
print(f"\nSaved raw pull to: {raw_path}")


# ----------------------------------------------------------------------
# STEP 2: Pull corporate actions — SPLITS ONLY (bonuses show up as
# "splits" in yfinance's data model, e.g. a 1:1 bonus appears as a
# split ratio of 2.0 — yfinance doesn't distinguish bonus vs true split,
# which is fine for adjustment purposes since the MATH is identical:
# both mean "more shares exist now for the same underlying value,
# so historical prices need to be scaled down to be comparable.")
#
# We deliberately do NOT pull .dividends here — see the scope
# discussion: this session is price-return adjustment only.
# ----------------------------------------------------------------------

split_frames = []

for ticker in pilot_tickers:
    print(f"Pulling split history for {ticker}...")

    t = yf.Ticker(ticker)
    splits = t.splits  # returns a pandas Series: index=date, value=split ratio

    if splits.empty:
        print(f"  No splits/bonuses recorded for {ticker} in yfinance's history.")
        continue

    # Convert the Series to a tidy DataFrame so we can stack tickers together
    s_df = splits.reset_index()
    s_df.columns = ['split_date', 'split_ratio']

    # FIX (found in first run): yfinance's split dates come back TIMEZONE-AWARE
    # (e.g. 2024-10-28 00:00:00+05:30, Indian Standard Time), but our price
    # panel's Date column is timezone-NAIVE. Comparing an aware timestamp to
    # a naive one raises a TypeError in pandas (correctly — it refuses to
    # guess which timezone the naive side "means"). We strip the timezone
    # here so both sides of every later comparison are naive and comparable.
    # NOTE: this assumes the underlying calendar date is what matters (not
    # the intraday time), which is true for split effective dates.
    s_df['split_date'] = s_df['split_date'].dt.tz_localize(None)

    s_df['ticker'] = ticker
    split_frames.append(s_df)

    print(f"  Found {len(s_df)} split/bonus event(s):")
    print(s_df.to_string(index=False))

if split_frames:
    corp_actions = pd.concat(split_frames, ignore_index=True)
else:
    corp_actions = pd.DataFrame(columns=['split_date', 'split_ratio', 'ticker'])
    print("\n[WARNING] No split/bonus events found for ANY of the 6 pilot tickers.")
    print("This would be surprising for large, long-listed companies — investigate")
    print("before trusting the adjustment step below.")

ca_path = BASE + f'corporate_actions/yfinance_splits_pilot_v1_{today_str}.parquet'
corp_actions.to_parquet(ca_path, index=False)
print(f"\nSaved corporate actions to: {ca_path}")


# ----------------------------------------------------------------------
# STEP 3: Apply our OWN split adjustment — the core logic of this session.
#
# THE MATH, explained:
# A split/bonus with ratio R means "each old share becomes R new shares."
# yfinance's split_ratio IS already this R (e.g. a 1:1 bonus -> ratio 2.0,
# a 2-for-1 split -> ratio 2.0, a 1:2 bonus -> ratio 1.5).
#
# To make historical prices comparable to TODAY's share count, every
# price BEFORE the split date must be DIVIDED by the cumulative product
# of all split ratios that happened AFTER that date (working backward
# from today). Volume gets the OPPOSITE treatment — MULTIPLIED by the
# same cumulative ratio, since share count went up by that factor.
#
# We build this as an explicit, step-by-step multiplier column so you
# can see exactly what factor was applied to every single date —
# nothing is hidden inside a single opaque function call.
# ----------------------------------------------------------------------

def apply_split_adjustment(price_df: pd.DataFrame, splits_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """
    Adjusts one ticker's raw OHLCV for splits/bonuses.

    Parameters
    ----------
    price_df : raw OHLCV rows for a SINGLE ticker, sorted by date ascending
    splits_df : split events for that SAME ticker (date, ratio)
    ticker : ticker string, used only for logging/error messages

    Returns
    -------
    price_df with new columns:
        - adj_factor : the cumulative multiplier applied to THIS row's price
        - open_adj, high_adj, low_adj, close_adj, volume_adj : adjusted fields
    """
    df = price_df.sort_values('Date').reset_index(drop=True).copy()

    # Start every row's adjustment factor at 1.0 (no adjustment yet).
    # We will multiply this down for every split that happened AFTER
    # that row's date, working from the most recent split backward.
    df['adj_factor'] = 1.0

    if splits_df.empty:
        # No splits at all for this ticker — adj_factor stays 1.0 everywhere,
        # meaning raw price IS the adjusted price. Log this so it's visible,
        # not silently assumed.
        print(f"  [{ticker}] No splits found — adjusted price equals raw price for all dates.")
    else:
        # Sort splits from MOST RECENT to OLDEST, because we apply them
        # working backward through time (each split affects all dates
        # strictly before its own effective date).
        splits_sorted = splits_df.sort_values('split_date', ascending=False)

        cumulative_factor = 1.0
        for _, row in splits_sorted.iterrows():
            split_date = row['split_date']
            ratio = row['split_ratio']

            # This split's ratio gets folded into the cumulative factor,
            # and applies to every date STRICTLY BEFORE split_date.
            cumulative_factor *= ratio

            mask = df['Date'] < split_date
            df.loc[mask, 'adj_factor'] = cumulative_factor

            print(f"  [{ticker}] Applying split ratio {ratio} effective {split_date.date()}: "
                  f"{mask.sum()} prior rows now scaled by cumulative factor {cumulative_factor}")

    # Apply the factor: PRICES divide by adj_factor, VOLUME multiplies by it.
    # (Price divides because more shares exist for the same value -> lower
    #  historical price per share. Volume multiplies for the mirror reason —
    #  the same historical trade would represent more shares under today's count.)
    for col in ['Open', 'High', 'Low', 'Close']:
        df[col.lower() + '_adj'] = df[col] / df['adj_factor']
    df['volume_adj'] = df['Volume'] * df['adj_factor']

    return df


# Run the adjustment, ticker by ticker, and stack results back together.
adjusted_frames = []
for ticker in pilot_tickers:
    price_sub = raw_panel[raw_panel['ticker'] == ticker].copy()
    if price_sub.empty:
        continue
    splits_sub = corp_actions[corp_actions['ticker'] == ticker].copy()

    adj_sub = apply_split_adjustment(price_sub, splits_sub, ticker)
    adjusted_frames.append(adj_sub)

adjusted_panel = pd.concat(adjusted_frames, ignore_index=True)

adj_path = BASE + 'adjusted/prices_adjusted_pilot_v1.parquet'
adjusted_panel.to_parquet(adj_path, index=False)
print(f"\nSaved adjusted panel to: {adj_path}")


# ----------------------------------------------------------------------
# STEP 4: VALIDATION — do not just claim this worked, show it.
#
# (a) Pick the ticker/date with the biggest single-day RAW return —
#     likely a split event — and print raw vs adjusted price around it.
#     We should see a JUMP in raw close, and a SMOOTH line in adjusted close.
# (b) Cross-check our adjusted close against yfinance's OWN
#     auto_adjust=True close, as an independent sanity check (not as
#     ground truth — yfinance's own adjustment might itself be
#     imperfect, this is just "do we land in the same neighborhood").
# ----------------------------------------------------------------------

print("\n" + "="*70)
print("VALIDATION")
print("="*70)

raw_panel_sorted = raw_panel.sort_values(['ticker', 'Date'])
raw_panel_sorted['raw_ret'] = raw_panel_sorted.groupby('ticker')['Close'].pct_change()

biggest_moves = raw_panel_sorted.reindex(
    raw_panel_sorted['raw_ret'].abs().sort_values(ascending=False).index
).head(5)
print("\nTop 5 single-day RAW return magnitudes across the pilot (candidates")
print("for a split/bonus event we should now see corrected in adjusted data):")
print(biggest_moves[['ticker', 'Date', 'Close', 'raw_ret']].to_string(index=False))

if not biggest_moves.empty:
    check_ticker = biggest_moves.iloc[0]['ticker']
    check_date = biggest_moves.iloc[0]['Date']

    window = adjusted_panel[
        (adjusted_panel['ticker'] == check_ticker) &
        (adjusted_panel['Date'] >= check_date - pd.Timedelta(days=5)) &
        (adjusted_panel['Date'] <= check_date + pd.Timedelta(days=5))
    ]
    print(f"\n--- Raw vs adjusted close around the biggest move: {check_ticker}, {check_date.date()} ---")
    print(window[['Date', 'Close', 'close_adj', 'adj_factor']].to_string(index=False))
    print("\n>>> CHECK: does 'Close' show a jump here, while 'close_adj' is smooth?")
    print(">>> If yes, our adjustment is working as intended.")
    print(">>> If 'close_adj' ALSO jumps, something is wrong with the adjustment logic.")

    print("\n--- Cross-check vs yfinance's own auto_adjust=True close ---")
    yf_adjusted = yf.download(check_ticker, start=START, end=END, auto_adjust=True, progress=False)
    # Same MultiIndex fix as Step 1 — this is a single-ticker download too,
    # and yfinance applies the same column structure regardless of context.
    if isinstance(yf_adjusted.columns, pd.MultiIndex):
        yf_adjusted.columns = yf_adjusted.columns.get_level_values(0)
    yf_adjusted = yf_adjusted.reset_index()

    compare = adjusted_panel[adjusted_panel['ticker'] == check_ticker][['Date', 'close_adj']].merge(
        yf_adjusted[['Date', 'Close']].rename(columns={'Close': 'yf_auto_adjusted_close'}),
        on='Date', how='inner'
    )
    compare['pct_diff'] = (
        (compare['close_adj'] - compare['yf_auto_adjusted_close']).abs()
        / compare['yf_auto_adjusted_close'] * 100
    )

    print(compare.describe())
    print(f"\nMax % difference between OUR adjustment and yfinance's own: {compare['pct_diff'].max():.4f}%")
    print(">>> Small differences are expected (yfinance's auto_adjust may ALSO fold in")
    print(">>> dividend adjustment even though we did not — worth checking if this gap")
    print(">>> is non-trivial, since it would mean the two series are not directly comparable).")
    print(">>> Large/systematic differences would mean our split logic has a bug.")
else:
    print("\n[WARNING] No price data to validate against — raw_panel may be empty.")

print("\n" + "="*70)
print("SESSION 1 PILOT COMPLETE — paste full output back for review")
print("="*70)

Mounted at /content/drive
Folder structure created under: /content/drive/MyDrive/moonshot_compounding/
 - /content/drive/MyDrive/moonshot_compounding/raw/universe
 - /content/drive/MyDrive/moonshot_compounding/raw/prices
 - /content/drive/MyDrive/moonshot_compounding/corporate_actions
 - /content/drive/MyDrive/moonshot_compounding/adjusted
 - /content/drive/MyDrive/moonshot_compounding/universe
 - /content/drive/MyDrive/moonshot_compounding/features
 - /content/drive/MyDrive/moonshot_compounding/validation
Pulling raw OHLCV for RELIANCE.NS...
Pulling raw OHLCV for HDFCBANK.NS...
Pulling raw OHLCV for INFY.NS...
Pulling raw OHLCV for TCS.NS...
Pulling raw OHLCV for ITC.NS...
Pulling raw OHLCV for WIPRO.NS...

Raw pull complete. Shape: (8874, 8)
Price       Date   Adj Close       Close        High         Low        Open  \
0     2015-01-01  189.125381  202.959000  203.896194  201.987518  202.593277   
1     2015-01-02  188.624786  202.421829  204.821960  202.136108  203.004715   
2     

In [2]:
# ======================================================================
# SESSION 1 — DIAGNOSTIC FOLLOW-UP
# Purpose: (a) validate adjustment against an ACTUAL known split date,
#          not the "biggest mover" (which turned out not to be a split),
#          (b) investigate whether the ~45-79% gap vs yfinance's own
#          adjustment is a smooth dividend-like trend or a sudden jump
#          (which would indicate a missed corporate action).
#
# Assumes raw_panel, corp_actions, and adjusted_panel already exist in
# this Colab session from the previous cell. If you've restarted the
# runtime, re-run the Session 1 script first.
# ======================================================================

import pandas as pd

# ------------------------------------------------------------
# PART A: Validate against a REAL, KNOWN split date this time.
# Picking WIPRO.NS 2019-03-06 (ratio 1.333333) — squarely inside
# our 2015-2020 pilot window, unlike 2024-12-03 which is outside it.
# ------------------------------------------------------------
print("="*70)
print("PART A: VALIDATION AGAINST AN ACTUAL SPLIT DATE (not biggest-mover)")
print("="*70)

check_ticker = 'WIPRO.NS'
check_date = pd.Timestamp('2019-03-06')

window = adjusted_panel[
    (adjusted_panel['ticker'] == check_ticker) &
    (adjusted_panel['Date'] >= check_date - pd.Timedelta(days=5)) &
    (adjusted_panel['Date'] <= check_date + pd.Timedelta(days=5))
]
print(f"\n--- Raw vs adjusted close around REAL split date: {check_ticker}, {check_date.date()} (ratio 1.333333) ---")
print(window[['Date', 'Close', 'close_adj', 'adj_factor']].to_string(index=False))
print("\n>>> EXPECTED: 'Close' jumps DOWN by roughly 1/1.333 = 25% right at this date")
print(">>> (more shares issued -> lower price per share, no real value change).")
print(">>> 'close_adj' and 'adj_factor' should show the SMOOTH, corrected series.")


# ------------------------------------------------------------
# PART B: Is the gap vs yfinance's own adjustment SMOOTH (dividend-like)
# or does it JUMP at a specific date (missed corporate action)?
#
# We look at the RATIO (not just % diff) between our close_adj and
# yfinance's auto-adjusted close, day by day, and print it at regular
# intervals across the whole window. A smoothly drifting ratio supports
# "dividends we didn't adjust for." A sudden step-change at one specific
# date supports "we (or yfinance) are missing a real corporate action
# at that date."
# ------------------------------------------------------------
print("\n" + "="*70)
print("PART B: SHAPE OF THE DIVERGENCE OVER TIME")
print("="*70)

import yfinance as yf

yf_auto = yf.download(check_ticker, start='2015-01-01', end='2020-12-31',
                       auto_adjust=True, progress=False)
if isinstance(yf_auto.columns, pd.MultiIndex):
    yf_auto.columns = yf_auto.columns.get_level_values(0)
yf_auto = yf_auto.reset_index()

our_series = adjusted_panel[adjusted_panel['ticker'] == check_ticker][['Date', 'close_adj']]
compare = our_series.merge(
    yf_auto[['Date', 'Close']].rename(columns={'Close': 'yf_auto_close'}),
    on='Date', how='inner'
).sort_values('Date').reset_index(drop=True)

# Ratio, not percent difference — easier to see if it's drifting smoothly
# or stepping suddenly.
compare['ratio_yf_over_ours'] = compare['yf_auto_close'] / compare['close_adj']

# Print every ~60th row (roughly quarterly) so we can see the TREND
# across the whole window without a 1479-row wall of numbers.
sampled = compare.iloc[::60]
print("\nRatio of (yfinance's own adjusted close) / (our adjusted close),")
print("sampled roughly quarterly across the whole window:\n")
print(sampled[['Date', 'close_adj', 'yf_auto_close', 'ratio_yf_over_ours']].to_string(index=False))

print("\n>>> READ THIS AS: if 'ratio_yf_over_ours' drifts SMOOTHLY and GRADUALLY")
print(">>> (e.g. 1.1 -> 1.3 -> 1.5 -> 1.8 as we go further back in time), that is")
print(">>> consistent with a cumulative dividend-reinvestment effect we deliberately")
print(">>> did not include.")
print(">>>")
print(">>> if instead the ratio JUMPS suddenly at one specific date and is roughly")
print(">>> FLAT before and after that jump, that means there is a real corporate")
print(">>> action at that date that our split-only adjustment MISSED entirely —")
print(">>> for example, a bonus issue that yfinance's .splits didn't capture, or")
print(">>> a demerger/spinoff, which is a fundamentally different kind of event")
print(">>> from a simple split and needs separate handling.")

print(f"\nRatio range: min={compare['ratio_yf_over_ours'].min():.3f}, "
      f"max={compare['ratio_yf_over_ours'].max():.3f}")

print("\n" + "="*70)
print("DIAGNOSTIC COMPLETE — paste full output back for review")
print("="*70)

PART A: VALIDATION AGAINST AN ACTUAL SPLIT DATE (not biggest-mover)

--- Raw vs adjusted close around REAL split date: WIPRO.NS, 2019-03-06 (ratio 1.333333) ---
      Date      Close  close_adj  adj_factor
2019-03-01 140.887527  52.832823    2.666667
2019-03-05 136.350037  51.131264    2.666667
2019-03-06 138.725006  69.362503    2.000000
2019-03-07 134.425003  67.212502    2.000000
2019-03-08 128.850006  64.425003    2.000000
2019-03-11 129.875000  64.937500    2.000000

>>> EXPECTED: 'Close' jumps DOWN by roughly 1/1.333 = 25% right at this date
>>> (more shares issued -> lower price per share, no real value change).
>>> 'close_adj' and 'adj_factor' should show the SMOOTH, corrected series.

PART B: SHAPE OF THE DIVERGENCE OVER TIME

Ratio of (yfinance's own adjusted close) / (our adjusted close),
sampled roughly quarterly across the whole window:

      Date  close_adj  yf_auto_close  ratio_yf_over_ours
2015-01-01  19.420318      91.154816            4.693786
2015-03-31  22.108013  

In [3]:
# ======================================================================
# SESSION 1 — DIRECT TEST: is yfinance's "Close" (auto_adjust=False)
# already split-adjusted, contrary to our assumption?
#
# Logic: at each REAL split date, compute the ACTUAL % change in raw
# Close from the day before to the day of/after. Compare that to the
# THEORETICAL % change we'd expect if Close were genuinely unadjusted
# (a price should drop by (1 - 1/ratio) * 100%, purely mechanically,
# with no real value change).
#
# If actual change ~= theoretical expected drop  -> Close IS raw, as assumed.
# If actual change ~= 0% (no drop at all)        -> Close is ALREADY adjusted.
# ======================================================================

import pandas as pd

wipro_splits = corp_actions[corp_actions['ticker'] == 'WIPRO.NS'].sort_values('split_date')
wipro_prices = raw_panel[raw_panel['ticker'] == 'WIPRO.NS'].sort_values('Date').reset_index(drop=True)

print("="*70)
print("DIRECT TEST: raw Close behavior at each real WIPRO split date")
print("="*70)

for _, row in wipro_splits.iterrows():
    split_date = row['split_date']
    ratio = row['split_ratio']

    # Find the last trading day strictly BEFORE the split, and the split
    # date itself (or the next available trading day on/after it).
    before = wipro_prices[wipro_prices['Date'] < split_date].tail(1)
    on_or_after = wipro_prices[wipro_prices['Date'] >= split_date].head(1)

    if before.empty or on_or_after.empty:
        print(f"\nSplit {split_date.date()} (ratio {ratio}): outside our pulled date range, skipping.")
        continue

    close_before = before['Close'].values[0]
    close_after = on_or_after['Close'].values[0]
    actual_pct_change = (close_after - close_before) / close_before * 100

    # If genuinely unadjusted, price should mechanically fall by this much:
    theoretical_pct_change = (1 - 1/ratio) * 100 * -1  # negative = a drop

    print(f"\nSplit date: {split_date.date()}, ratio: {ratio}")
    print(f"  Close before: {close_before:.2f}  ({before['Date'].values[0]})")
    print(f"  Close after:  {close_after:.2f}  ({on_or_after['Date'].values[0]})")
    print(f"  ACTUAL % change:      {actual_pct_change:+.2f}%")
    print(f"  THEORETICAL % change if Close were truly RAW: {theoretical_pct_change:+.2f}%")

    if abs(actual_pct_change) < abs(theoretical_pct_change) / 3:
        print(f"  >>> VERDICT: Close shows almost NO drop -> Close appears ALREADY split-adjusted.")
    elif abs(actual_pct_change - theoretical_pct_change) < 5:
        print(f"  >>> VERDICT: Close matches the expected raw drop -> Close IS genuinely unadjusted.")
    else:
        print(f"  >>> VERDICT: UNCLEAR — actual change doesn't cleanly match either scenario, investigate further.")

print("\n" + "="*70)
print("TEST COMPLETE — paste full output back")
print("="*70)

DIRECT TEST: raw Close behavior at each real WIPRO split date

Split 1997-10-20 (ratio 3.0): outside our pulled date range, skipping.

Split 1999-09-27 (ratio 5.0): outside our pulled date range, skipping.

Split 2004-06-25 (ratio 3.0): outside our pulled date range, skipping.

Split 2005-08-22 (ratio 2.0): outside our pulled date range, skipping.

Split 2010-06-15 (ratio 1.6666666666666667): outside our pulled date range, skipping.

Split date: 2017-06-13, ratio: 2.0
  Close before: 98.69  (2017-06-12T00:00:00.000000000)
  Close after:  97.13  (2017-06-13T00:00:00.000000000)
  ACTUAL % change:      -1.59%
  THEORETICAL % change if Close were truly RAW: -50.00%
  >>> VERDICT: Close shows almost NO drop -> Close appears ALREADY split-adjusted.

Split date: 2019-03-06, ratio: 1.3333333333333333
  Close before: 136.35  (2019-03-05T00:00:00.000000000)
  Close after:  138.73  (2019-03-06T00:00:00.000000000)
  ACTUAL % change:      +1.74%
  THEORETICAL % change if Close were truly RAW: -25.0

In [4]:
# ======================================================================
# SESSION 1 — DIRECT TEST, GENERALIZED: confirm on a second (and third)
# ticker that raw "Close" (auto_adjust=False) is already split-adjusted.
#
# Same logic as the WIPRO-only version, but wrapped in a function so we
# can run it against INFY.NS and TCS.NS without copy-pasting the loop.
# ======================================================================

import pandas as pd

def test_split_adjustment_already_applied(ticker: str, raw_panel: pd.DataFrame, corp_actions: pd.DataFrame):
    """
    For every split event we have on file for `ticker`, compares the
    ACTUAL % change in raw Close across the split boundary against the
    THEORETICAL % change we'd expect if Close were genuinely unadjusted.

    Prints a verdict per split. Does not return anything — this is a
    diagnostic, not a data transformation step.
    """
    splits = corp_actions[corp_actions['ticker'] == ticker].sort_values('split_date')
    prices = raw_panel[raw_panel['ticker'] == ticker].sort_values('Date').reset_index(drop=True)

    print("="*70)
    print(f"DIRECT TEST: raw Close behavior at each real {ticker} split date")
    print("="*70)

    if splits.empty:
        print(f"\nNo splits on file for {ticker} — nothing to test.")
        return

    for _, row in splits.iterrows():
        split_date = row['split_date']
        ratio = row['split_ratio']

        before = prices[prices['Date'] < split_date].tail(1)
        on_or_after = prices[prices['Date'] >= split_date].head(1)

        if before.empty or on_or_after.empty:
            print(f"\nSplit {split_date.date()} (ratio {ratio}): outside our pulled date range, skipping.")
            continue

        close_before = before['Close'].values[0]
        close_after = on_or_after['Close'].values[0]
        actual_pct_change = (close_after - close_before) / close_before * 100

        theoretical_pct_change = (1 - 1/ratio) * 100 * -1  # negative = a drop

        print(f"\nSplit date: {split_date.date()}, ratio: {ratio}")
        print(f"  Close before: {close_before:.2f}  ({before['Date'].values[0]})")
        print(f"  Close after:  {close_after:.2f}  ({on_or_after['Date'].values[0]})")
        print(f"  ACTUAL % change:      {actual_pct_change:+.2f}%")
        print(f"  THEORETICAL % change if Close were truly RAW: {theoretical_pct_change:+.2f}%")

        if abs(actual_pct_change) < abs(theoretical_pct_change) / 3:
            print(f"  >>> VERDICT: Close shows almost NO drop -> Close appears ALREADY split-adjusted.")
        elif abs(actual_pct_change - theoretical_pct_change) < 5:
            print(f"  >>> VERDICT: Close matches the expected raw drop -> Close IS genuinely unadjusted.")
        else:
            print(f"  >>> VERDICT: UNCLEAR — actual change doesn't cleanly match either scenario, investigate further.")


# ------------------------------------------------------------
# Run against INFY.NS (2 splits inside our 2015-2020 window:
# 2015-06-15 ratio 2.0, and 2018-09-04 ratio 2.0) and TCS.NS
# (1 split inside our window: 2018-05-31 ratio 2.0).
# Both use raw_panel / corp_actions already in memory from Session 1.
# ------------------------------------------------------------

test_split_adjustment_already_applied('INFY.NS', raw_panel, corp_actions)
print()
test_split_adjustment_already_applied('TCS.NS', raw_panel, corp_actions)

print("\n" + "="*70)
print("CONFIRMATION TEST COMPLETE — paste full output back")
print("="*70)

DIRECT TEST: raw Close behavior at each real INFY.NS split date

Split 1997-08-19 (ratio 2.0): outside our pulled date range, skipping.

Split 1999-02-10 (ratio 2.0): outside our pulled date range, skipping.

Split 2000-01-24 (ratio 2.0): outside our pulled date range, skipping.

Split 2000-01-27 (ratio 2.0): outside our pulled date range, skipping.

Split 2004-07-01 (ratio 4.0): outside our pulled date range, skipping.

Split 2006-07-13 (ratio 2.0): outside our pulled date range, skipping.

Split 2014-12-02 (ratio 2.0): outside our pulled date range, skipping.

Split date: 2015-06-15, ratio: 2.0
  Close before: 493.76  (2015-06-12T00:00:00.000000000)
  Close after:  495.23  (2015-06-15T00:00:00.000000000)
  ACTUAL % change:      +0.30%
  THEORETICAL % change if Close were truly RAW: -50.00%
  >>> VERDICT: Close shows almost NO drop -> Close appears ALREADY split-adjusted.

Split date: 2018-09-04, ratio: 2.0
  Close before: 717.12  (2018-09-03T00:00:00.000000000)
  Close after:  737.15

In [5]:
# ======================================================================
# WIDENED PILOT — add smaller/less liquid names to stress-test whether
# "Close is already split-adjusted" holds beyond mega-cap names.
#
# Picked from Sheet4's June 2025 cohort (smaller, more recently-added
# names) rather than more large-caps like our first 6.
#
# Assumes raw_panel, corp_actions, and the test_split_adjustment_already_applied
# function are already in memory from earlier cells. If not, re-run
# those cells first.
# ======================================================================

import yfinance as yf
import pandas as pd
from datetime import datetime

new_tickers = ['GRAVITA.NS', 'NEULANDLAB.NS', 'LTFOODS.NS']

# Use a WIDER date range than the original pilot (2010-present) since
# these smaller names may have had splits at any point in their history,
# and we want the best chance of catching a real event to test against.
START = '2010-01-01'
END = '2026-07-01'

new_raw_frames = []
for ticker in new_tickers:
    print(f"Pulling raw OHLCV for {ticker}...")
    df = yf.download(ticker, start=START, end=END, auto_adjust=False, progress=False)

    if df.empty:
        print(f"  [WARNING] No data returned for {ticker} — investigate before proceeding.")
        continue

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df['ticker'] = ticker
    df = df.reset_index()
    new_raw_frames.append(df)

if not new_raw_frames:
    raise RuntimeError("No data returned for ANY of the widened pilot tickers — stop and investigate.")

new_raw_panel = pd.concat(new_raw_frames, ignore_index=True)
print(f"\nWidened raw pull complete. Shape: {new_raw_panel.shape}")

# Pull split history for the new tickers, same tz-fix as before
new_split_frames = []
for ticker in new_tickers:
    print(f"Pulling split history for {ticker}...")
    t = yf.Ticker(ticker)
    splits = t.splits

    if splits.empty:
        print(f"  No splits/bonuses recorded for {ticker}.")
        continue

    s_df = splits.reset_index()
    s_df.columns = ['split_date', 'split_ratio']
    s_df['split_date'] = s_df['split_date'].dt.tz_localize(None)
    s_df['ticker'] = ticker
    new_split_frames.append(s_df)
    print(f"  Found {len(s_df)} split/bonus event(s):")
    print(s_df.to_string(index=False))

if new_split_frames:
    new_corp_actions = pd.concat(new_split_frames, ignore_index=True)
else:
    new_corp_actions = pd.DataFrame(columns=['split_date', 'split_ratio', 'ticker'])
    print("\n[NOTE] None of the 3 widened-pilot tickers have any recorded splits.")
    print("This means we can't stress-test the split-adjustment assumption on")
    print("these specific names — pick different tickers with known splits if")
    print("this happens, rather than treating 'no data' as a pass.")

# Save the widened pull too, timestamped, same convention as before
today_str = datetime.now().strftime('%Y-%m-%d')
BASE = '/content/drive/MyDrive/moonshot_compounding/'
new_raw_panel.to_parquet(BASE + f'raw/prices/yfinance_raw_widened_pilot_v1_{today_str}.parquet', index=False)
new_corp_actions.to_parquet(BASE + f'corporate_actions/yfinance_splits_widened_pilot_v1_{today_str}.parquet', index=False)

# ------------------------------------------------------------
# Run the SAME direct test as before, on these new tickers.
# Reuses the function defined earlier in this Colab session —
# NOT redefined here, to avoid two copies of the same logic
# silently drifting apart over time.
# ------------------------------------------------------------
print("\n" + "="*70)
print("RUNNING DIRECT SPLIT TEST ON WIDENED PILOT TICKERS")
print("="*70)

for ticker in new_tickers:
    test_split_adjustment_already_applied(ticker, new_raw_panel, new_corp_actions)
    print()

print("="*70)
print("WIDENED PILOT TEST COMPLETE — paste full output back")
print("="*70)

Pulling raw OHLCV for GRAVITA.NS...
Pulling raw OHLCV for NEULANDLAB.NS...
Pulling raw OHLCV for LTFOODS.NS...

Widened raw pull complete. Shape: (11998, 8)
Pulling split history for GRAVITA.NS...
  Found 1 split/bonus event(s):
split_date  split_ratio     ticker
2012-05-25          5.0 GRAVITA.NS
Pulling split history for NEULANDLAB.NS...
  No splits/bonuses recorded for NEULANDLAB.NS.
Pulling split history for LTFOODS.NS...
  Found 1 split/bonus event(s):
split_date  split_ratio     ticker
2017-02-07         10.0 LTFOODS.NS

RUNNING DIRECT SPLIT TEST ON WIDENED PILOT TICKERS
DIRECT TEST: raw Close behavior at each real GRAVITA.NS split date

Split date: 2012-05-25, ratio: 5.0
  Close before: 180.05  (2012-05-24T00:00:00.000000000)
  Close after:  189.00  (2012-05-25T00:00:00.000000000)
  ACTUAL % change:      +4.97%
  THEORETICAL % change if Close were truly RAW: -80.00%
  >>> VERDICT: Close shows almost NO drop -> Close appears ALREADY split-adjusted.

DIRECT TEST: raw Close behavio

In [6]:
# ======================================================================
# STAGE 3A — Load or build the canonical PIT universe file.
#
# This file was originally built and inspected in a separate sandbox
# during our earlier diagnostic sessions. It has NOT yet been built
# inside THIS Colab environment. This cell checks if it already exists
# on Drive (e.g. if you've run this before) and builds it fresh from
# the Bloomberg Sheet4 Excel file if not.
# ======================================================================
import pandas as pd
import numpy as np
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
UNIVERSE_PATH = BASE + 'universe/canonical_membership_semiannual.parquet'

if os.path.exists(UNIVERSE_PATH):
    print(f"Found existing canonical universe file at {UNIVERSE_PATH} — loading it.")
    canonical = pd.read_parquet(UNIVERSE_PATH)
else:
    print("Canonical universe file not found on Drive — building it now from Sheet4.")
    print("You will be prompted to upload NSE_historical_constituents.xlsx.")
    from google.colab import files
    uploaded = files.upload()
    xlsx_path = list(uploaded.keys())[0]

    import openpyxl
    import datetime as dt

    wb = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb['Sheet4']
    rows = list(ws.iter_rows(values_only=True))
    header_row = rows[1]
    # Sheet4 is a wide, block-structured layout: every 6th column starts a
    # new snapshot date block (date, id(), name(), id_isin(), id_exch_symbol()).
    date_positions = [(i, v) for i, v in enumerate(header_row) if isinstance(v, dt.datetime)]

    records = []
    for offset, snap_date in date_positions:
        id_col, name_col, isin_col, exch_col = offset + 1, offset + 2, offset + 3, offset + 4
        for r in rows[2:]:
            if id_col >= len(r):
                continue
            bbg_id = r[id_col]
            if bbg_id is None:
                continue
            records.append((
                snap_date,
                bbg_id,
                r[name_col] if name_col < len(r) else None,
                r[isin_col] if isin_col < len(r) else None,
                r[exch_col] if exch_col < len(r) else None,
            ))

    df = pd.DataFrame(records, columns=['snap_date', 'bbg_id', 'name', 'isin', 'exch_symbol'])
    df['snap_date'] = pd.to_datetime(df['snap_date'])

    # KNOWN ISSUE: the 2018-06-30 snapshot in the raw file is truncated
    # (353 rows instead of ~500 — a partial-pull artifact, confirmed by
    # checking it starts alphabetically at "ABAN", missing names like
    # "3M", "360ONE", "63MOONS" present in every neighboring snapshot).
    # We substitute the 2017-12-31 snapshot as a documented stand-in.
    # Quantified risk (see build_log.md): comparing the two real snapshots
    # bracketing this gap (2017-12-31 vs 2018-12-31) shows 56 additions +
    # 56 removals combined across BOTH 2018 rebalances — an upper bound of
    # ~22% of the universe potentially misclassified specifically in the
    # 2018-07-01 to 2018-12-30 window. This is a real, non-trivial risk,
    # not a rounding footnote — flagged explicitly here and in build_log.md
    # so any future test result concentrated in that window gets scrutiny.
    snap_2017_12 = df[df['snap_date'] == pd.Timestamp('2017-12-31')].copy()
    carried = snap_2017_12.copy()
    carried['snap_date'] = pd.Timestamp('2018-06-30')
    carried['snapshot_source'] = 'carried_forward_from_2017-12-31'

    df_clean = df[df['snap_date'] != pd.Timestamp('2018-06-30')].copy()
    df_clean['snapshot_source'] = 'observed'

    canonical = pd.concat([df_clean, carried], ignore_index=True)
    canonical = canonical.sort_values(['snap_date', 'isin']).reset_index(drop=True)

    canonical.to_parquet(UNIVERSE_PATH, index=False)
    print(f"Built and saved canonical universe to {UNIVERSE_PATH}")

print(f"\nCanonical universe loaded: {canonical.shape[0]} rows, "
      f"{canonical['isin'].nunique()} unique ISINs, "
      f"{canonical['snap_date'].nunique()} snapshot dates.")


# ======================================================================
# STAGE 3B — Derive the pull ticker list from the canonical universe.
#
# We confirmed earlier that every ISIN in this dataset maps to exactly
# ONE exch_symbol across all 33 snapshots — no ticker-rename cases exist.
# That means the crosswalk is simple: one ISIN -> one ticker.
#
# TWO THINGS DISCOVERED WHILE TESTING THIS LOGIC (both fixed below):
#
# 1. Missing symbols are stored as the LITERAL STRING "#N/A" in this
#    file, not as a true null. pandas' isna()/dropna() do NOT catch a
#    string that merely SAYS "#N/A" — we have to check for both forms
#    explicitly, or 5 bogus rows silently collapse into one fake
#    "#N/A.NS" ticker and corrupt the pull list with duplicates.
#
# 2. A stock can have a valid symbol on MOST snapshots but "#N/A" on
#    just one or two (e.g. 360 ONE WAM Ltd — formerly IIFL Wealth
#    Management — shows "#N/A" only during its 2023 rebrand window,
#    with a valid "360ONE" symbol on 11 of its 13 appearances). Such a
#    stock is NOT actually unpullable — we must only exclude an ISIN if
#    it has NO valid symbol on ANY snapshot, not if it merely has a gap
#    on some snapshots.
# ======================================================================

is_missing_row = canonical['exch_symbol'].isna() | (canonical['exch_symbol'] == '#N/A')

# Pullable: the most recent VALID (non-missing) symbol per ISIN,
# ignoring any "#N/A"/null rows entirely when picking the latest one.
isin_symbol_map = (
    canonical[~is_missing_row]
    .sort_values('snap_date')
    .groupby('isin')
    .last()[['exch_symbol', 'name']]
    .reset_index()
)
isin_symbol_map['yf_ticker'] = isin_symbol_map['exch_symbol'].str.strip() + '.NS'

# Truly excluded: ISINs with NO valid symbol on ANY snapshot at all —
# i.e. ISINs that never made it into isin_symbol_map above. This is
# NOT the same as "ISINs with a missing symbol on at least one date."
all_isins = set(canonical['isin'].unique())
pullable_isins = set(isin_symbol_map['isin'])
truly_excluded_isins = all_isins - pullable_isins

excluded = canonical[canonical['isin'].isin(truly_excluded_isins)][['isin', 'name']].drop_duplicates()
excluded['reason'] = 'no_valid_exch_symbol_on_any_snapshot_likely_unlisted_or_newly_demerged_entity'
os.makedirs(BASE + 'validation', exist_ok=True)
excluded.to_csv(BASE + 'validation/excluded_tickers_no_symbol.csv', index=False)

print(f"\nDerived {len(isin_symbol_map)} pullable tickers from Sheet4.")
print(f"Excluded {len(excluded)} ISINs with NO valid symbol on any snapshot:")
print(excluded.to_string(index=False))

dupes = isin_symbol_map[isin_symbol_map['yf_ticker'].duplicated(keep=False)]
assert dupes.empty, f"Found duplicate yf_tickers — investigate before proceeding:\n{dupes}"
assert len(isin_symbol_map) + len(excluded) == canonical['isin'].nunique(), \
    "Pullable + excluded counts don't reconcile with total unique ISINs — stop and investigate"
print("\n[PASS] Pullable + excluded counts reconcile exactly with total unique ISINs. No duplicate tickers.")

pull_list = isin_symbol_map[['isin', 'yf_ticker', 'name']].copy()
pull_list.to_csv(BASE + 'universe/pull_ticker_list_v1.csv', index=False)
print(f"\nSaved pull list ({len(pull_list)} tickers) to universe/pull_ticker_list_v1.csv")
print("\nSample of pull list:")
print(pull_list.head(10).to_string(index=False))

Found existing canonical universe file at /content/drive/MyDrive/moonshot_compounding/universe/canonical_membership_semiannual.parquet — loading it.

Canonical universe loaded: 16507 rows, 1076 unique ISINs, 33 snapshot dates.

Derived 1071 pullable tickers from Sheet4.
Excluded 5 ISINs with NO valid symbol on any snapshot:
        isin                        name                                                                        reason
INE2KCE01013    Kwality Wall's India Ltd no_valid_exch_symbol_on_any_snapshot_likely_unlisted_or_newly_demerged_entity
INE1CDF01017 Vedanta Aluminium Metal Ltd no_valid_exch_symbol_on_any_snapshot_likely_unlisted_or_newly_demerged_entity
INE1CLE01013  Vedanta Iron and Steel Ltd no_valid_exch_symbol_on_any_snapshot_likely_unlisted_or_newly_demerged_entity
INE694L01019           Vedanta Power Ltd no_valid_exch_symbol_on_any_snapshot_likely_unlisted_or_newly_demerged_entity
INE704J01044            Malco Energy Ltd no_valid_exch_symbol_on_any_snapshot_l

In [7]:
# ======================================================================
# STAGE 3C — Bulk pull: raw OHLCV + corporate actions for the full
# universe (1,071 tickers), 2010-01-01 to present.
#
# DESIGN, explained:
#
# - One ticker = one atomic unit of work. Each ticker's price data is
#   saved to its OWN small parquet file (raw/prices/batches/<ticker>.parquet),
#   and its OWN split data likewise (corporate_actions/batches/<ticker>.parquet).
#   This means a crash or Colab disconnect never corrupts a big shared
#   file — at worst we lose whatever ticker was mid-download when it happened.
#
# - A checkpoint CSV (_pull_progress.csv) records every ticker's outcome:
#   success/failed, row count, error message, timestamp. Re-running this
#   cell SKIPS any ticker already marked 'success' — so this script is
#   safely re-runnable as many times as needed until everything succeeds,
#   without ever re-downloading data we already have.
#
# - Each ticker gets up to 3 attempts (1 initial + 2 retries) with a pause
#   between retries, before being logged as a genuine failure. Genuine
#   failures are NOT silently dropped — they stay visible in the
#   checkpoint log with their error message, for us to investigate.
#
# - A deliberate 0.75s delay between tickers reduces the risk of Yahoo
#   rate-limiting us partway through a run of 1,000+ tickers.
#
# EXPECTED RUNTIME: with ~1,071 tickers and a ~1-2s round trip each
# (download + delay), this will likely take 30-60+ minutes. That's fine —
# because of the checkpointing, you can stop and resume this at any time
# by just re-running the cell.
# ======================================================================

import yfinance as yf
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime

BASE = '/content/drive/MyDrive/moonshot_compounding/'

START = '2010-01-01'
END = pd.Timestamp.today().strftime('%Y-%m-%d')  # pull through today

PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
CA_BATCH_DIR = BASE + 'corporate_actions/batches/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'

os.makedirs(PRICE_BATCH_DIR, exist_ok=True)
os.makedirs(CA_BATCH_DIR, exist_ok=True)

DELAY_SECONDS = 0.75        # deliberate pause between EVERY ticker
MAX_ATTEMPTS = 3            # 1 initial attempt + 2 retries
RETRY_DELAY_SECONDS = 5     # longer pause specifically before a retry
CHECKPOINT_FLUSH_EVERY = 25 # save progress to disk every N tickers


# ------------------------------------------------------------
# Load or initialize the checkpoint log.
# ------------------------------------------------------------
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = pd.read_csv(CHECKPOINT_PATH)
    print(f"Resuming from existing checkpoint: {len(checkpoint)} tickers already attempted.")
else:
    checkpoint = pd.DataFrame(columns=['ticker', 'status', 'rows_pulled', 'n_splits', 'error_message', 'timestamp'])
    print("No existing checkpoint found — starting fresh.")

already_succeeded = set(checkpoint[checkpoint['status'] == 'success']['ticker'])
print(f"{len(already_succeeded)} tickers already successfully pulled — these will be SKIPPED.\n")

pull_list_df = pd.read_csv(BASE + 'universe/pull_ticker_list_v1.csv')
tickers_to_pull = [t for t in pull_list_df['yf_ticker'].tolist() if t not in already_succeeded]

print(f"{len(tickers_to_pull)} tickers remaining to pull this run.\n")

new_checkpoint_rows = []


# ------------------------------------------------------------
# Main pull loop.
# ------------------------------------------------------------
for i, ticker in enumerate(tickers_to_pull):
    success = False
    last_error = None

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            df = yf.download(ticker, start=START, end=END, auto_adjust=False, progress=False)

            if df.empty:
                raise ValueError("yfinance returned an empty DataFrame (ticker may be delisted, "
                                  "merged, or an invalid/incorrect symbol)")

            # Same MultiIndex-flatten fix confirmed necessary in the pilot —
            # every single-ticker yf.download() call needs this.
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df['ticker'] = ticker
            df = df.reset_index()

            safe_name = ticker.replace('.', '_')
            df.to_parquet(PRICE_BATCH_DIR + f'{safe_name}.parquet', index=False)

            # Corporate actions for the same ticker, same successful attempt.
            # Kept on file even though we don't need it for price adjustment
            # (confirmed this session: yfinance's Close is already split-
            # adjusted) — useful later if a total-return series is ever needed.
            t_obj = yf.Ticker(ticker)
            splits = t_obj.splits
            n_splits = 0
            if not splits.empty:
                s_df = splits.reset_index()
                s_df.columns = ['split_date', 'split_ratio']
                s_df['split_date'] = s_df['split_date'].dt.tz_localize(None)
                s_df['ticker'] = ticker
                s_df.to_parquet(CA_BATCH_DIR + f'{safe_name}.parquet', index=False)
                n_splits = len(s_df)

            new_checkpoint_rows.append({
                'ticker': ticker, 'status': 'success', 'rows_pulled': len(df),
                'n_splits': n_splits, 'error_message': '', 'timestamp': datetime.now().isoformat(),
            })
            success = True
            break  # do not retry after a genuine success

        except Exception as e:
            last_error = str(e)
            if attempt < MAX_ATTEMPTS:
                print(f"  [{ticker}] attempt {attempt} failed ({last_error}) — "
                      f"retrying after {RETRY_DELAY_SECONDS}s...")
                time.sleep(RETRY_DELAY_SECONDS)

    if not success:
        print(f"  [{ticker}] FAILED after {MAX_ATTEMPTS} attempts: {last_error}")
        new_checkpoint_rows.append({
            'ticker': ticker, 'status': 'failed', 'rows_pulled': 0,
            'n_splits': 0, 'error_message': last_error, 'timestamp': datetime.now().isoformat(),
        })

    # Periodic checkpoint flush: never lose more than CHECKPOINT_FLUSH_EVERY
    # tickers of progress if the runtime disconnects mid-run.
    if (i + 1) % CHECKPOINT_FLUSH_EVERY == 0 or (i + 1) == len(tickers_to_pull):
        print(f"Progress: {i + 1}/{len(tickers_to_pull)} tickers processed this run.")
        checkpoint = pd.concat([checkpoint, pd.DataFrame(new_checkpoint_rows)], ignore_index=True)
        checkpoint.to_csv(CHECKPOINT_PATH, index=False)
        new_checkpoint_rows = []

    time.sleep(DELAY_SECONDS)

# Final safety flush in case the last partial batch wasn't a multiple of
# CHECKPOINT_FLUSH_EVERY (already covered by the (i+1)==len(...) condition
# above, but kept here as a defensive no-op in case that branch is ever
# changed later).
if new_checkpoint_rows:
    checkpoint = pd.concat([checkpoint, pd.DataFrame(new_checkpoint_rows)], ignore_index=True)
    checkpoint.to_csv(CHECKPOINT_PATH, index=False)


# ------------------------------------------------------------
# Run summary.
#
# IMPORTANT: the checkpoint file is APPEND-ONLY — every run adds new
# rows rather than overwriting old ones. That means a ticker that
# failed in an earlier run and succeeded on a later retry will have
# MULTIPLE rows in the checkpoint (one 'failed', one 'success'). This
# is deliberately kept as a full audit trail of every attempt — but it
# means we must summarize by each ticker's LATEST status, not by raw
# row counts, or a ticker that eventually succeeded would still show
# up as "failed" in this summary just because of a stale earlier row.
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("BULK PULL RUN COMPLETE (this session)")
print("=" * 70)

latest_status = (
    checkpoint.sort_values('timestamp')
    .groupby('ticker')
    .last()[['status', 'error_message']]
    .reset_index()
)

final_success = (latest_status['status'] == 'success').sum()
final_failed = (latest_status['status'] == 'failed').sum()
print(f"Total successful (by latest attempt per ticker): {final_success} / {len(pull_list_df)}")
print(f"Total still failed (by latest attempt per ticker): {final_failed}")

if final_failed > 0:
    print("\nTickers CURRENTLY failed (latest attempt only — investigate before treating the pull as complete):")
    print(latest_status[latest_status['status'] == 'failed'][['ticker', 'error_message']].to_string(index=False))

print("\nIf tickers remain unattempted or failed, just RE-RUN this cell — it will")
print("automatically skip anything already marked 'success' and retry the rest.")

[long output removed for the public repo]


In [8]:
# ======================================================================
# STAGE 3C-RETRY — for tickers STILL failing after simply re-running
# Stage 3C once (which should have already cleared transient failures
# like rate-limiting — confirmed SPICEJET.NS was one such false failure).
#
# This does something Stage 3C's plain retry CANNOT do: tries a .BO
# (BSE) fallback for tickers that fail under .NS. Many NSE-delisted
# companies remained listed on BSE, and Yahoo often retains that price
# history under the .BO suffix even after NSE delisting.
#
# Anything that STILL fails after the .BO fallback gets written to a
# separate review file — these are the ones we treat as probable real
# delistings/mergers UNLESS manually checked otherwise (e.g. the way we
# just confirmed SPICEJET.NS was a false failure via a web search).
# ======================================================================

import yfinance as yf
import pandas as pd
import os
import time
from datetime import datetime

BASE = '/content/drive/MyDrive/moonshot_compounding/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
CA_BATCH_DIR = BASE + 'corporate_actions/batches/'

START = '2010-01-01'
END = pd.Timestamp.today().strftime('%Y-%m-%d')
DELAY_SECONDS = 0.75

checkpoint = pd.read_csv(CHECKPOINT_PATH)
latest_status = (
    checkpoint.sort_values('timestamp')
    .groupby('ticker')
    .last()[['status', 'error_message']]
    .reset_index()
)
currently_failed = latest_status[latest_status['status'] == 'failed']['ticker'].tolist()

print(f"{len(currently_failed)} tickers currently failed — attempting .BO fallback for each.\n")

new_checkpoint_rows = []
still_unresolved = []

for i, ns_ticker in enumerate(currently_failed):
    bo_ticker = ns_ticker.replace('.NS', '.BO')

    try:
        df = yf.download(bo_ticker, start=START, end=END, auto_adjust=False, progress=False)

        if df.empty:
            raise ValueError(f".BO fallback also returned empty for {bo_ticker}")

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # IMPORTANT: we save this under the ORIGINAL .NS-derived filename,
        # but tag which actual source ticker was used, so downstream
        # consumers of this file know it came from BSE, not NSE — this
        # matters if NSE and BSE prices ever diverge slightly (they can,
        # due to different liquidity/trading patterns on each exchange).
        df['ticker'] = ns_ticker              # keep original identity for joins
        df['source_ticker'] = bo_ticker        # but record the ACTUAL pull source
        df['exchange_source'] = 'BSE_fallback'
        df = df.reset_index()

        safe_name = ns_ticker.replace('.', '_')
        df.to_parquet(PRICE_BATCH_DIR + f'{safe_name}.parquet', index=False)

        print(f"  [{ns_ticker}] RECOVERED via BSE fallback ({bo_ticker}), {len(df)} rows.")
        new_checkpoint_rows.append({
            'ticker': ns_ticker, 'status': 'success', 'rows_pulled': len(df),
            'n_splits': 0, 'error_message': 'recovered_via_BSE_fallback',
            'timestamp': datetime.now().isoformat(),
        })

    except Exception as e:
        print(f"  [{ns_ticker}] still unresolved — .BO fallback also failed: {e}")
        still_unresolved.append({'ticker': ns_ticker, 'bo_attempt_error': str(e)})
        new_checkpoint_rows.append({
            'ticker': ns_ticker, 'status': 'failed', 'rows_pulled': 0,
            'n_splits': 0, 'error_message': f'NS and BO both failed: {e}',
            'timestamp': datetime.now().isoformat(),
        })

    time.sleep(DELAY_SECONDS)

checkpoint = pd.concat([checkpoint, pd.DataFrame(new_checkpoint_rows)], ignore_index=True)
checkpoint.to_csv(CHECKPOINT_PATH, index=False)

# Save the still-unresolved list explicitly for manual review — these are
# our candidate "genuine delisting/merger, accept as a real gap" tickers,
# but should be EYEBALLED (a few spot-checked via web search, as we did
# for SpiceJet) rather than assumed correct without any check at all.
unresolved_df = pd.DataFrame(still_unresolved)
unresolved_path = BASE + 'validation/unresolved_tickers_for_manual_review.csv'
unresolved_df.to_csv(unresolved_path, index=False)

print("\n" + "=" * 70)
print(f"RECOVERED via BSE fallback: {len(currently_failed) - len(still_unresolved)}")
print(f"STILL UNRESOLVED (saved to {unresolved_path}): {len(still_unresolved)}")
print("=" * 70)
if still_unresolved:
    print("\nUnresolved tickers (spot-check a sample of these before accepting as real gaps):")
    print(unresolved_df.to_string(index=False))

[long output removed for the public repo]


In [9]:
# ======================================================================
# STAGE 3-QA — Check the SUCCESSFUL pulls, not just the failed ones.
#
# A ticker can return a non-empty DataFrame (no exception raised) and
# STILL be wrong: sparse date coverage, duplicate rows, all-NaN or
# garbage values, a schema that silently drifted. "No error" is not
# the same as "correct" — this script checks for the specific failure
# modes that wouldn't have shown up as a crash.
# ======================================================================

import pandas as pd
import numpy as np
import os
import glob

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'

batch_files = glob.glob(PRICE_BATCH_DIR + '*.parquet')
print(f"Found {len(batch_files)} successfully-pulled ticker files to QA.\n")

qa_rows = []

for fpath in batch_files:
    ticker = os.path.basename(fpath).replace('_parquet', '').replace('.parquet', '').replace('_', '.', 1)
    # NOTE: the replace('_', '.', 1) above assumes tickers have exactly one
    # underscore standing in for the original '.' (e.g. RELIANCE_NS ->
    # RELIANCE.NS). This holds for plain '.NS'/'.BO' tickers but would
    # break if a ticker itself ever contained an underscore — worth
    # re-checking if that ever comes up, not assumed silently correct here.

    df = pd.read_parquet(fpath)

    n_rows = len(df)
    n_duplicate_dates = df['Date'].duplicated().sum() if 'Date' in df.columns else np.nan
    date_min = df['Date'].min() if 'Date' in df.columns and n_rows > 0 else None
    date_max = df['Date'].max() if 'Date' in df.columns and n_rows > 0 else None

    # % NaN in each core OHLCV column
    core_cols = [c for c in ['Open', 'High', 'Low', 'Close', 'Volume'] if c in df.columns]
    pct_nan = {c: df[c].isna().mean() * 100 for c in core_cols}

    # Sanity: any non-positive Close or negative Volume? Should never happen
    # in legitimate price data.
    n_nonpositive_close = (df['Close'] <= 0).sum() if 'Close' in df.columns else np.nan
    n_negative_volume = (df['Volume'] < 0).sum() if 'Volume' in df.columns else np.nan

    # Expected trading days is roughly 252/year. A HUGE gap between actual
    # row count and (date_max - date_min) in calendar days, adjusted for
    # weekends/holidays, suggests missing chunks of history rather than
    # a company that simply listed late.
    if date_min is not None and date_max is not None:
        calendar_days_span = (date_max - date_min).days
        expected_trading_days_approx = calendar_days_span * (252 / 365)
        coverage_ratio = n_rows / expected_trading_days_approx if expected_trading_days_approx > 0 else np.nan
    else:
        coverage_ratio = np.nan

    qa_rows.append({
        'ticker': ticker,
        'n_rows': n_rows,
        'date_min': date_min,
        'date_max': date_max,
        'n_duplicate_dates': n_duplicate_dates,
        'pct_nan_close': pct_nan.get('Close', np.nan),
        'pct_nan_volume': pct_nan.get('Volume', np.nan),
        'n_nonpositive_close': n_nonpositive_close,
        'n_negative_volume': n_negative_volume,
        'coverage_ratio': coverage_ratio,  # ~1.0 = dense, well below 1.0 = gappy
    })

qa_df = pd.DataFrame(qa_rows)
qa_path = BASE + 'validation/successful_pull_qa_report.csv'
qa_df.to_csv(qa_path, index=False)

print("=" * 70)
print("QA SUMMARY")
print("=" * 70)

print(f"\nTotal tickers QA'd: {len(qa_df)}")

dupes_found = qa_df[qa_df['n_duplicate_dates'] > 0]
print(f"\nTickers with duplicate dates within their own file: {len(dupes_found)}")
if not dupes_found.empty:
    print(dupes_found[['ticker', 'n_duplicate_dates']].to_string(index=False))

bad_close = qa_df[qa_df['n_nonpositive_close'] > 0]
print(f"\nTickers with non-positive Close values (should be ZERO, ever): {len(bad_close)}")
if not bad_close.empty:
    print(bad_close[['ticker', 'n_nonpositive_close']].to_string(index=False))

bad_volume = qa_df[qa_df['n_negative_volume'] > 0]
print(f"\nTickers with negative Volume values (should be ZERO, ever): {len(bad_volume)}")
if not bad_volume.empty:
    print(bad_volume[['ticker', 'n_negative_volume']].to_string(index=False))

high_nan = qa_df[qa_df['pct_nan_close'] > 5]
print(f"\nTickers with >5% NaN in Close price: {len(high_nan)}")
if not high_nan.empty:
    print(high_nan[['ticker', 'pct_nan_close']].to_string(index=False))

sparse = qa_df[qa_df['coverage_ratio'] < 0.85]
print(f"\nTickers with sparse coverage (<85% of expected trading days present — "
      f"possible large gaps in the middle of their history, not just a late listing): {len(sparse)}")
if not sparse.empty:
    print(sparse[['ticker', 'n_rows', 'date_min', 'date_max', 'coverage_ratio']]
          .sort_values('coverage_ratio').to_string(index=False))

print(f"\nFull QA report saved to: {qa_path}")
print("\nNOTE: a LOW coverage_ratio is expected and FINE for a company that")
print("listed after 2010 (it just has less history, not gappy history) — check")
print("date_min for those before treating them as a real problem. The concerning")
print("case is a coverage_ratio well below 1.0 for a stock with an early date_min,")
print("which points to actual missing chunks mid-history, not a late listing.")

Found 964 successfully-pulled ticker files to QA.

QA SUMMARY

Total tickers QA'd: 964

Tickers with duplicate dates within their own file: 0

Tickers with non-positive Close values (should be ZERO, ever): 0

Tickers with negative Volume values (should be ZERO, ever): 0

Tickers with >5% NaN in Close price: 0

Tickers with sparse coverage (<85% of expected trading days present — possible large gaps in the middle of their history, not just a late listing): 1
     ticker  n_rows   date_min   date_max  coverage_ratio
KENNAMET.NS    1129 2019-08-19 2026-07-10        0.649685

Full QA report saved to: /content/drive/MyDrive/moonshot_compounding/validation/successful_pull_qa_report.csv

NOTE: a LOW coverage_ratio is expected and FINE for a company that
listed after 2010 (it just has less history, not gappy history) — check
date_min for those before treating them as a real problem. The concerning
case is a coverage_ratio well below 1.0 for a stock with an early date_min,
which points to actua

In [10]:
# ======================================================================
# STAGE 3-QA2 — Cross-validate price history against the PIT universe.
#
# WHY THIS CHECK EXISTS (separate from the earlier QA pass):
# The earlier QA only checked each ticker's data against ITSELF (is its
# own date range internally dense). It could NOT catch a ticker whose
# entire history silently starts too late — e.g. if a company renamed
# and yfinance treats the new ticker as a fresh listing rather than
# continuing the old symbol's price history. That would "succeed" with
# no error, and look internally fine, while actually missing years of
# real, needed history.
#
# This check instead compares each ticker's ACTUAL price start date
# against the TRUE first-appearance date recorded for its ISIN in our
# authoritative canonical PIT universe file (built from Bloomberg
# Sheet4) — this is the one thing we have that tells us independently
# when a company should have data, rather than trusting the price file
# to judge itself.
# ======================================================================

import pandas as pd
import numpy as np
import glob
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
UNIVERSE_PATH = BASE + 'universe/canonical_membership_semiannual.parquet'

canonical = pd.read_parquet(UNIVERSE_PATH)

# Same "#N/A" handling as Stage 3B — must stay consistent with how the
# pull list was originally derived, or this check would compare against
# the wrong reference for the excluded/edge-case ISINs.
is_missing_row = canonical['exch_symbol'].isna() | (canonical['exch_symbol'] == '#N/A')
valid_membership = canonical[~is_missing_row].copy()

# For each ISIN: the ticker we actually pulled (latest valid symbol) AND
# the TRUE first appearance date across its ENTIRE history under that
# ISIN — not just the most recent symbol era, but from whenever this
# ISIN first shows up as a constituent at all.
isin_summary = (
    valid_membership.sort_values('snap_date')
    .groupby('isin')
    .agg(exch_symbol=('exch_symbol', 'last'),
         true_first_appearance=('snap_date', 'min'))
    .reset_index()
)
isin_summary['ticker'] = isin_summary['exch_symbol'].str.strip() + '.NS'

# Load actual date_min for every successfully pulled ticker.
batch_files = glob.glob(PRICE_BATCH_DIR + '*.parquet')
price_start_rows = []
for fpath in batch_files:
    ticker = os.path.basename(fpath).replace('.parquet', '').replace('_', '.', 1)
    df = pd.read_parquet(fpath, columns=['Date'])
    if df.empty:
        continue
    price_start_rows.append({'ticker': ticker, 'actual_price_start': df['Date'].min()})

price_starts = pd.DataFrame(price_start_rows)

# The actual cross-validation.
merged = price_starts.merge(
    isin_summary[['ticker', 'true_first_appearance']],
    on='ticker', how='left'
)
merged['gap_days'] = (merged['actual_price_start'] - merged['true_first_appearance']).dt.days

# A ticker's price data STARTING LATER than its true first appearance as
# a constituent is the concerning case (missing real history). A price
# series starting EARLIER than PIT-constituent-entry is fine and
# expected (a company can trade for years before entering the Nifty 500).
FLAG_THRESHOLD_DAYS = 180  # more than ~6 months late suggests a real gap, not vendor noise

no_reference = merged[merged['true_first_appearance'].isna()]
flagged = merged[merged['gap_days'] > FLAG_THRESHOLD_DAYS].sort_values('gap_days', ascending=False)

print("=" * 70)
print("CROSS-VALIDATION: price start date vs. true PIT first-appearance date")
print("=" * 70)

print(f"\nTotal successfully-pulled tickers checked: {len(merged)}")
print(f"Tickers with NO matching ISIN reference found (investigate separately "
      f"— likely a join mismatch, not a real problem, but worth confirming): {len(no_reference)}")
if not no_reference.empty:
    print(no_reference[['ticker']].to_string(index=False))

print(f"\nTickers whose price data starts >{FLAG_THRESHOLD_DAYS} days AFTER their "
      f"true first appearance as a constituent (SUSPECTED SILENT GAP): {len(flagged)}")
if not flagged.empty:
    print(flagged[['ticker', 'true_first_appearance', 'actual_price_start', 'gap_days']]
          .to_string(index=False))
else:
    print("None found — no evidence of the 'renamed ticker silently truncates history' failure mode.")

qa2_path = BASE + 'validation/pit_crossvalidation_report.csv'
merged.to_csv(qa2_path, index=False)
print(f"\nFull cross-validation report saved to: {qa2_path}")

CROSS-VALIDATION: price start date vs. true PIT first-appearance date

Total successfully-pulled tickers checked: 964
Tickers with NO matching ISIN reference found (investigate separately — likely a join mismatch, not a real problem, but worth confirming): 0

Tickers whose price data starts >180 days AFTER their true first appearance as a constituent (SUSPECTED SILENT GAP): 8
       ticker true_first_appearance actual_price_start  gap_days
 BALLARPUR.NS            2010-06-30         2026-07-10      5854
 PUNJLLOYD.NS            2010-06-30         2026-07-10      5854
IVRCLINFRA.NS            2010-06-30         2026-07-10      5854
  ERAINFRA.NS            2010-06-30         2026-07-10      5854
      GSPL.NS            2010-06-30         2026-05-11      5794
 MOSERBAER.NS            2010-06-30         2018-03-14      2814
   INNOIND.NS            2012-12-31         2018-03-14      1899
  GEODESIC.NS            2013-06-30         2018-03-29      1733

Full cross-validation report saved 

In [11]:
import os
import pandas as pd

BASE = '/content/drive/MyDrive/moonshot_compounding/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'

bad_ticker = 'HINDMOTORS.NS'

# Delete the useless 1-row file so it can't be mistaken for real data later
bad_file = PRICE_BATCH_DIR + bad_ticker.replace('.', '_') + '.parquet'
if os.path.exists(bad_file):
    os.remove(bad_file)
    print(f"Removed {bad_file} (only 1 row — not usable).")

checkpoint = pd.read_csv(CHECKPOINT_PATH)
new_row = pd.DataFrame([{
    'ticker': bad_ticker, 'status': 'failed', 'rows_pulled': 0,
    'n_splits': 0, 'error_message': 'recovered_via_BSE_but_only_1_row_treated_as_unusable',
    'timestamp': pd.Timestamp.now().isoformat(),
}])
checkpoint = pd.concat([checkpoint, new_row], ignore_index=True)
checkpoint.to_csv(CHECKPOINT_PATH, index=False)
print("Checkpoint updated — HINDMOTORS.NS now correctly shows as failed/unusable.")

Checkpoint updated — HINDMOTORS.NS now correctly shows as failed/unusable.


In [12]:
import pandas as pd
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
CV_REPORT_PATH = BASE + 'validation/pit_crossvalidation_report.csv'
UNRESOLVED_PATH = BASE + 'validation/unresolved_tickers_for_manual_review.csv'

FLAG_THRESHOLD_DAYS = 180

cv = pd.read_csv(CV_REPORT_PATH)
flagged = cv[cv['gap_days'] > FLAG_THRESHOLD_DAYS].copy()

print(f"Reclassifying {len(flagged)} tickers from 'success' to 'failed' — "
      f"they have real gaps against the authoritative PIT universe.\n")

checkpoint = pd.read_csv(CHECKPOINT_PATH)
new_rows = []

for _, row in flagged.iterrows():
    ticker = row['ticker']
    gap = row['gap_days']

    # Distinguish the two patterns for the log, since they have different
    # implications: Pattern A (phantom quote) vs Pattern B (real but late data).
    fpath = PRICE_BATCH_DIR + ticker.replace('.', '_') + '.parquet'
    if os.path.exists(fpath):
        df = pd.read_parquet(fpath)
        n_rows = len(df)
    else:
        n_rows = 0

    if n_rows <= 2:
        reason = f'phantom_single_row_quote_gap_{int(gap)}_days'
        if os.path.exists(fpath):
            os.remove(fpath)
            print(f"  [{ticker}] Removed phantom {n_rows}-row file. Gap: {int(gap)} days.")
    else:
        reason = f'real_data_but_starts_{int(gap)}_days_after_true_PIT_entry'
        print(f"  [{ticker}] Flagged as incomplete (kept file for reference, "
              f"{n_rows} rows, but starts {int(gap)} days late).")

    new_rows.append({
        'ticker': ticker, 'status': 'failed', 'rows_pulled': 0 if n_rows <= 2 else n_rows,
        'n_splits': 0, 'error_message': reason, 'timestamp': pd.Timestamp.now().isoformat(),
    })

checkpoint = pd.concat([checkpoint, pd.DataFrame(new_rows)], ignore_index=True)
checkpoint.to_csv(CHECKPOINT_PATH, index=False)

# Merge into the same unresolved-for-review file as the original 130,
# so we have ONE consolidated list of "needs Bloomberg or manual fix"
# tickers, not two separate scattered lists.
if os.path.exists(UNRESOLVED_PATH):
    existing_unresolved = pd.read_csv(UNRESOLVED_PATH)
else:
    existing_unresolved = pd.DataFrame(columns=['ticker', 'bo_attempt_error'])

newly_unresolved = flagged[['ticker']].copy()
newly_unresolved['bo_attempt_error'] = 'failed_pit_crossvalidation_see_gap_days_in_cv_report'

combined_unresolved = pd.concat([existing_unresolved, newly_unresolved], ignore_index=True).drop_duplicates(subset='ticker')
combined_unresolved.to_csv(UNRESOLVED_PATH, index=False)

print(f"\nConsolidated unresolved list now has {len(combined_unresolved)} tickers total "
      f"(saved to {UNRESOLVED_PATH}).")

# Final true count
latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last()['status']
true_success_count = (latest_status == 'success').sum()
print(f"\nTRUE current successful ticker count after reclassification: {true_success_count}")

Reclassifying 8 tickers from 'success' to 'failed' — they have real gaps against the authoritative PIT universe.

  [GEODESIC.NS] Flagged as incomplete (kept file for reference, 114 rows, but starts 1733 days late).
  [INNOIND.NS] Flagged as incomplete (kept file for reference, 76 rows, but starts 1899 days late).
  [MOSERBAER.NS] Flagged as incomplete (kept file for reference, 366 rows, but starts 2814 days late).
  [BALLARPUR.NS] Removed phantom 1-row file. Gap: 5854 days.
  [ERAINFRA.NS] Removed phantom 1-row file. Gap: 5854 days.
  [GSPL.NS] Removed phantom 1-row file. Gap: 5794 days.
  [IVRCLINFRA.NS] Removed phantom 1-row file. Gap: 5854 days.
  [PUNJLLOYD.NS] Removed phantom 1-row file. Gap: 5854 days.

Consolidated unresolved list now has 115 tickers total (saved to /content/drive/MyDrive/moonshot_compounding/validation/unresolved_tickers_for_manual_review.csv).

TRUE current successful ticker count after reclassification: 956


In [13]:
import pandas as pd

BASE = '/content/drive/MyDrive/moonshot_compounding/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'
UNRESOLVED_PATH = BASE + 'validation/unresolved_tickers_for_manual_review.csv'
PULL_LIST_PATH = BASE + 'universe/pull_ticker_list_v1.csv'

checkpoint = pd.read_csv(CHECKPOINT_PATH)
latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last()['status']

unresolved_file = pd.read_csv(UNRESOLVED_PATH)
pull_list = pd.read_csv(PULL_LIST_PATH)

print("="*70)
print("RECONCILIATION: checkpoint (source of truth) vs unresolved CSV file")
print("="*70)

# 1. Does every ticker in the pull list have a checkpoint entry at all?
missing_from_checkpoint = set(pull_list['yf_ticker']) - set(latest_status.index)
print(f"\nTickers in pull list with NO checkpoint entry at all: {len(missing_from_checkpoint)}")
if missing_from_checkpoint:
    print(sorted(missing_from_checkpoint))

# 2. Does the unresolved CSV contain any ticker that checkpoint says is ACTUALLY successful?
stale_in_unresolved = [
    t for t in unresolved_file['ticker']
    if t in latest_status.index and latest_status[t] == 'success'
]
print(f"\nTickers in unresolved_tickers_for_manual_review.csv that checkpoint "
      f"says are ACTUALLY successful (stale entries): {len(stale_in_unresolved)}")
if stale_in_unresolved:
    print(sorted(stale_in_unresolved))

# 3. Does checkpoint have 'failed' tickers that are MISSING from the unresolved CSV?
checkpoint_failed = set(latest_status[latest_status == 'failed'].index)
unresolved_set = set(unresolved_file['ticker'])
missing_from_unresolved = checkpoint_failed - unresolved_set
print(f"\nTickers checkpoint says are 'failed' but are MISSING from the unresolved CSV: "
      f"{len(missing_from_unresolved)}")
if missing_from_unresolved:
    print(sorted(missing_from_unresolved))

# 4. The TRUE, trustworthy numbers, straight from checkpoint only.
true_success = (latest_status == 'success').sum()
true_failed = (latest_status == 'failed').sum()
print(f"\n--- TRUE STATE (from checkpoint only) ---")
print(f"Success: {true_success}")
print(f"Failed: {true_failed}")
print(f"Total accounted for: {true_success + true_failed} (pull list has {len(pull_list)})")

RECONCILIATION: checkpoint (source of truth) vs unresolved CSV file

Tickers in pull list with NO checkpoint entry at all: 0

Tickers in unresolved_tickers_for_manual_review.csv that checkpoint says are ACTUALLY successful (stale entries): 0

Tickers checkpoint says are 'failed' but are MISSING from the unresolved CSV: 0

--- TRUE STATE (from checkpoint only) ---
Success: 956
Failed: 115
Total accounted for: 1071 (pull list has 1071)


In [14]:
import pandas as pd

BASE = '/content/drive/MyDrive/moonshot_compounding/'
checkpoint = pd.read_csv(BASE + 'raw/prices/_pull_progress.csv')
latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last().reset_index()

failed_now = latest_status[latest_status['status'] == 'failed'][['ticker', 'error_message']]
failed_now.to_csv(BASE + 'validation/unresolved_tickers_for_manual_review.csv', index=False)
print(f"Regenerated unresolved list from checkpoint: {len(failed_now)} tickers.")

Regenerated unresolved list from checkpoint: 115 tickers.


In [15]:
# ======================================================================
# Generate the Bloomberg pull list for the 116 unresolved tickers.
#
# IMPORTANT: uses bbg_id (e.g. "RANBAXY IS Equity"), NOT the yfinance
# .NS ticker — Bloomberg won't recognize .NS-style strings, and bbg_id
# is the identifier we already confirmed works (it's what Sheet4 itself
# was built from).
# ======================================================================

import pandas as pd

BASE = '/content/drive/MyDrive/moonshot_compounding/'

canonical = pd.read_parquet(BASE + 'universe/canonical_membership_semiannual.parquet')
checkpoint = pd.read_csv(BASE + 'raw/prices/_pull_progress.csv')

latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last().reset_index()
unresolved_tickers = latest_status[latest_status['status'] == 'failed']['ticker'].tolist()

print(f"{len(unresolved_tickers)} unresolved tickers to build a Bloomberg pull list for.")

# Rebuild the ISIN -> bbg_id mapping the same way we built ISIN -> yf_ticker
# in Stage 3B, so this stays consistent with how the original pull list
# was derived.
is_missing_row = canonical['exch_symbol'].isna() | (canonical['exch_symbol'] == '#N/A')
valid_membership = canonical[~is_missing_row]

isin_map = (
    valid_membership.sort_values('snap_date')
    .groupby('isin')
    .last()[['exch_symbol', 'bbg_id', 'name']]
    .reset_index()
)
isin_map['yf_ticker'] = isin_map['exch_symbol'].str.strip() + '.NS'

bloomberg_pull_list = isin_map[isin_map['yf_ticker'].isin(unresolved_tickers)][
    ['isin', 'yf_ticker', 'bbg_id', 'name']
].sort_values('bbg_id')

print(f"\nMatched {len(bloomberg_pull_list)} of {len(unresolved_tickers)} unresolved "
      f"tickers to a bbg_id.")
if len(bloomberg_pull_list) != len(unresolved_tickers):
    missing = set(unresolved_tickers) - set(bloomberg_pull_list['yf_ticker'])
    print(f"WARNING: {len(missing)} unresolved tickers had no bbg_id match — investigate: {missing}")

out_path = BASE + 'validation/bloomberg_pull_list_for_unresolved.csv'
bloomberg_pull_list.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path}")
print("\nJust the bbg_id column, ready to copy into Excel:\n")
print(bloomberg_pull_list['bbg_id'].to_string(index=False))

115 unresolved tickers to build a Bloomberg pull list for.

Matched 115 of 115 unresolved tickers to a bbg_id.

Saved to: /content/drive/MyDrive/moonshot_compounding/validation/bloomberg_pull_list_for_unresolved.csv

Just the bbg_id column, ready to copy into Excel:

1909315D IS Equity
1958850D IS Equity
2271068D IS Equity
2299371D IS Equity
    ABGS IS Equity
    ABNL IS Equity
 ACLGATI IS Equity
     ADV IS Equity
    ALBK IS Equity
    ALFA IS Equity
    ANDB IS Equity
    ASEL IS Equity
  BHAFIN IS Equity
    BILT IS Equity
    BINC IS Equity
    CAIR IS Equity
    CAST IS Equity
     CIS IS Equity
     CMC IS Equity
    COXK IS Equity
   CRPBK IS Equity
  DBNK/F IS Equity
    DDLL IS Equity
    DECH IS Equity
    DEWH IS Equity
   DHANI IS Equity
    DISH IS Equity
     EDA IS Equity
 EQUITAS IS Equity
    ERIE IS Equity
   ESOIL IS Equity
    ESRS IS Equity
    EXCC IS Equity
    FKOL IS Equity
GARWAOFF IS Equity
    GDPL IS Equity
     GEO IS Equity
    GEOD IS Equity
    GGAS I

In [17]:
# ======================================================================
# BLOOMBERG MERGE — parse the 116-ticker Bloomberg export, truncate
# trailing NaN rows to each ticker's real last-valid date, write
# per-ticker files matching the existing yfinance batch convention,
# and update the checkpoint.
#
# CONFIRMED WITH BLOOMBERG: this data is already split-adjusted.
# No custom adjustment step needed (unlike the earlier yfinance saga).
# ======================================================================

import pandas as pd
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'

# ------------------------------------------------------------
# STEP 1: Load the Bloomberg export. Upload it if prompted.
# ------------------------------------------------------------
from google.colab import files
print("Upload bloomberg_pull_list_for_unresolved.xlsx")
uploaded = files.upload()
xlsx_path = list(uploaded.keys())[0]

pull_list = pd.read_excel(xlsx_path, sheet_name='bloomberg_pull_list_for_unresol')
bbgid_to_ticker = dict(zip(pull_list['bbg_id'], pull_list['yf_ticker']))

df = pd.read_excel(xlsx_path, sheet_name='bloomberg_pull_list_for_unr (2)', header=[0, 1])
new_cols = list(df.columns)
new_cols[0] = ('DATE', '')  # rename the shared date column for a clean stack()
df.columns = pd.MultiIndex.from_tuples(new_cols)

df_long = df.set_index(('DATE', '')).stack(level=0, future_stack=True).reset_index()
df_long.columns = ['Date', 'bbg_id', 'Open', 'High', 'Low', 'Close', 'Volume']

# ------------------------------------------------------------
# STEP 2: Map bbg_id -> our canonical yf_ticker identity, so the WHOLE
# panel (yfinance + Bloomberg sourced) uses one consistent identity
# scheme regardless of which vendor actually supplied the data.
# ------------------------------------------------------------
df_long['ticker'] = df_long['bbg_id'].map(bbgid_to_ticker)

unmapped = df_long[df_long['ticker'].isna()]['bbg_id'].unique()
if len(unmapped) > 0:
    raise ValueError(f"{len(unmapped)} bbg_ids failed to map to a ticker — "
                      f"stop and investigate: {unmapped}")
print(f"All {df_long['bbg_id'].nunique()} bbg_ids mapped cleanly to tickers.")

df_long['source'] = 'bloomberg'
df_long = df_long.drop(columns=['bbg_id'])

# ------------------------------------------------------------
# STEP 3: Truncate trailing NaN rows per ticker.
#
# WHY: Bloomberg gave every security a row for every date in the shared
# grid, even long after it stopped trading — those rows are all-NaN,
# not real data. Left in place, they'd silently sit in the panel
# implying the security "exists" all the way to today. We truncate each
# ticker to its own true last-valid-Close date, and LOG the truncation
# explicitly rather than just dropping rows invisibly.
# ------------------------------------------------------------
truncation_log = []
cleaned_frames = []

for ticker, sub in df_long.groupby('ticker'):
    sub = sub.sort_values('Date').reset_index(drop=True)
    valid = sub.dropna(subset=['Close'])

    if valid.empty:
        truncation_log.append({
            'ticker': ticker, 'status': 'NO_VALID_DATA',
            'last_valid_date': None, 'rows_dropped': len(sub),
        })
        continue  # do not write a file for a fully-empty ticker

    last_valid_date = valid['Date'].max()
    kept = sub[sub['Date'] <= last_valid_date].copy()
    dropped_count = len(sub) - len(kept)

    truncation_log.append({
        'ticker': ticker, 'status': 'truncated' if dropped_count > 0 else 'no_truncation_needed',
        'last_valid_date': last_valid_date, 'rows_dropped': dropped_count,
    })
    cleaned_frames.append(kept)

truncation_df = pd.DataFrame(truncation_log)
truncation_path = BASE + 'validation/bloomberg_truncation_log.csv'
truncation_df.to_csv(truncation_path, index=False)

no_data_tickers = truncation_df[truncation_df['status'] == 'NO_VALID_DATA']['ticker'].tolist()
print(f"\n{len(no_data_tickers)} ticker(s) had NO valid data at all in the Bloomberg pull "
      f"(genuinely still unresolved): {no_data_tickers}")
print(f"Truncation log saved to: {truncation_path}")

# ------------------------------------------------------------
# STEP 4: Write one file per ticker, matching the EXACT filename
# convention already used for the yfinance batches.
# ------------------------------------------------------------
n_written = 0
for frame in cleaned_frames:
    ticker = frame['ticker'].iloc[0]
    safe_name = ticker.replace('.', '_')
    frame.to_parquet(PRICE_BATCH_DIR + f'{safe_name}.parquet', index=False)
    n_written += 1

print(f"\nWrote {n_written} Bloomberg-sourced ticker files to {PRICE_BATCH_DIR}")

# ------------------------------------------------------------
# STEP 5: Update the checkpoint. Tickers with real data -> success.
# The genuinely-empty ticker(s) stay 'failed', with a clear reason.
# ------------------------------------------------------------
checkpoint = pd.read_csv(CHECKPOINT_PATH)
new_rows = []

for _, row in truncation_df.iterrows():
    if row['status'] == 'NO_VALID_DATA':
        new_rows.append({
            'ticker': row['ticker'], 'status': 'failed', 'rows_pulled': 0,
            'n_splits': 0, 'error_message': 'bloomberg_pull_also_returned_no_valid_data',
            'timestamp': pd.Timestamp.now().isoformat(),
        })
    else:
        n_rows = pd.read_parquet(PRICE_BATCH_DIR + row['ticker'].replace('.', '_') + '.parquet').shape[0]
        new_rows.append({
            'ticker': row['ticker'], 'status': 'success', 'rows_pulled': n_rows,
            'n_splits': 0, 'error_message': 'recovered_via_bloomberg_terminal_pull',
            'timestamp': pd.Timestamp.now().isoformat(),
        })

checkpoint = pd.concat([checkpoint, pd.DataFrame(new_rows)], ignore_index=True)
checkpoint.to_csv(CHECKPOINT_PATH, index=False)

# ------------------------------------------------------------
# STEP 6: Regenerate the unresolved list FRESH from checkpoint
# (never hand-merge this file again — lesson learned earlier this session).
# ------------------------------------------------------------
latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last().reset_index()
still_failed = latest_status[latest_status['status'] == 'failed'][['ticker', 'error_message']]
still_failed.to_csv(BASE + 'validation/unresolved_tickers_for_manual_review.csv', index=False)

true_success = (latest_status['status'] == 'success').sum()
true_failed = (latest_status['status'] == 'failed').sum()

print(f"\n" + "=" * 70)
print("FINAL STATE AFTER BLOOMBERG MERGE")
print("=" * 70)
print(f"Success: {true_success}")
print(f"Still failed: {true_failed}")
print(f"Total: {true_success + true_failed} (should be 1071)")

Upload bloomberg_pull_list_for_unresolved.xlsx


Saving bloomberg_pull_list_for_unresolved.xlsx to bloomberg_pull_list_for_unresolved.xlsx
All 116 bbg_ids mapped cleanly to tickers.

1 ticker(s) had NO valid data at all in the Bloomberg pull (genuinely still unresolved): ['MIRCELECTR.NS']
Truncation log saved to: /content/drive/MyDrive/moonshot_compounding/validation/bloomberg_truncation_log.csv

Wrote 115 Bloomberg-sourced ticker files to /content/drive/MyDrive/moonshot_compounding/raw/prices/batches/

FINAL STATE AFTER BLOOMBERG MERGE
Success: 1070
Still failed: 1
Total: 1071 (should be 1071)


In [18]:
# ======================================================================
# STAGE 3-QA2 — Cross-validate price history against the PIT universe.
# (Same script as before — re-running now that Bloomberg-sourced files
# have been added to the batches folder. No changes needed; it globs
# the whole folder regardless of source.)
# ======================================================================

import pandas as pd
import numpy as np
import glob
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
UNIVERSE_PATH = BASE + 'universe/canonical_membership_semiannual.parquet'

canonical = pd.read_parquet(UNIVERSE_PATH)

is_missing_row = canonical['exch_symbol'].isna() | (canonical['exch_symbol'] == '#N/A')
valid_membership = canonical[~is_missing_row].copy()

isin_summary = (
    valid_membership.sort_values('snap_date')
    .groupby('isin')
    .agg(exch_symbol=('exch_symbol', 'last'),
         true_first_appearance=('snap_date', 'min'))
    .reset_index()
)
isin_summary['ticker'] = isin_summary['exch_symbol'].str.strip() + '.NS'

batch_files = glob.glob(PRICE_BATCH_DIR + '*.parquet')
price_start_rows = []
for fpath in batch_files:
    ticker = os.path.basename(fpath).replace('.parquet', '').replace('_', '.', 1)
    df = pd.read_parquet(fpath, columns=['Date'])
    if df.empty:
        continue
    price_start_rows.append({'ticker': ticker, 'actual_price_start': df['Date'].min()})

price_starts = pd.DataFrame(price_start_rows)

merged = price_starts.merge(
    isin_summary[['ticker', 'true_first_appearance']],
    on='ticker', how='left'
)
merged['gap_days'] = (merged['actual_price_start'] - merged['true_first_appearance']).dt.days

FLAG_THRESHOLD_DAYS = 180

no_reference = merged[merged['true_first_appearance'].isna()]
flagged = merged[merged['gap_days'] > FLAG_THRESHOLD_DAYS].sort_values('gap_days', ascending=False)

print("=" * 70)
print("CROSS-VALIDATION: price start date vs. true PIT first-appearance date")
print("=" * 70)

print(f"\nTotal successfully-pulled tickers checked: {len(merged)}")
print(f"Tickers with NO matching ISIN reference found: {len(no_reference)}")
if not no_reference.empty:
    print(no_reference[['ticker']].to_string(index=False))

print(f"\nTickers whose price data starts >{FLAG_THRESHOLD_DAYS} days AFTER their "
      f"true first appearance as a constituent (SUSPECTED SILENT GAP): {len(flagged)}")
if not flagged.empty:
    print(flagged[['ticker', 'true_first_appearance', 'actual_price_start', 'gap_days']]
          .to_string(index=False))
else:
    print("None found.")

qa2_path = BASE + 'validation/pit_crossvalidation_report.csv'
merged.to_csv(qa2_path, index=False)
print(f"\nFull cross-validation report saved to: {qa2_path}")

CROSS-VALIDATION: price start date vs. true PIT first-appearance date

Total successfully-pulled tickers checked: 1070
Tickers with NO matching ISIN reference found: 0

Tickers whose price data starts >180 days AFTER their true first appearance as a constituent (SUSPECTED SILENT GAP): 0
None found.

Full cross-validation report saved to: /content/drive/MyDrive/moonshot_compounding/validation/pit_crossvalidation_report.csv


In [19]:
# ======================================================================
# STAGE 3D — Consolidate all per-ticker batch files (yfinance + Bloomberg)
# into one final raw price panel, with schema harmonized across both
# sources, and a final QA pass on the fully merged result.
# ======================================================================

import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'

batch_files = glob.glob(PRICE_BATCH_DIR + '*.parquet')
print(f"Found {len(batch_files)} per-ticker batch files to consolidate.")

# ------------------------------------------------------------
# Schema harmonization: the 955 yfinance-sourced files carry an
# 'Adj Close' column (dividend-adjusted, NOT used for our price-return
# work) and no 'source' column. The 115 Bloomberg-sourced files carry
# 'source' but no 'Adj Close'. We standardize both onto one final schema
# so concatenation doesn't produce a sparse, misaligned table -- the
# exact bug we hit early on with yfinance's own MultiIndex columns.
# ------------------------------------------------------------
FINAL_COLS = ['Date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume',
              'adj_close_dividend_yfinance_unused', 'source']

def harmonize(df, default_source):
    df = df.copy()
    if 'source' not in df.columns:
        df['source'] = default_source
    if 'Adj Close' in df.columns:
        df = df.rename(columns={'Adj Close': 'adj_close_dividend_yfinance_unused'})
    else:
        df['adj_close_dividend_yfinance_unused'] = np.nan
    return df[FINAL_COLS]

frames = []
for fpath in batch_files:
    df = pd.read_parquet(fpath)
    # yfinance batch files never had a 'source' column written; Bloomberg
    # ones always do (added explicitly in the merge script). This default
    # correctly back-fills the 955 older files without needing to touch them.
    frames.append(harmonize(df, default_source='yfinance'))

full_panel = pd.concat(frames, ignore_index=True)
print(f"\nConsolidated panel shape: {full_panel.shape}")
print(f"Source breakdown:\n{full_panel.groupby('source')['ticker'].nunique()}")

# ------------------------------------------------------------
# Final QA pass on the FULLY MERGED result -- same checks as Stage 3-QA,
# run once more here because merging can itself introduce problems
# (e.g. a ticker accidentally represented in BOTH source files, which
# would show as duplicate (ticker, Date) rows here even if each
# individual file was clean on its own).
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL QA ON CONSOLIDATED PANEL")
print("=" * 70)

dupes = full_panel.duplicated(subset=['ticker', 'Date']).sum()
print(f"\nDuplicate (ticker, Date) rows across the WHOLE consolidated panel: {dupes}")
if dupes > 0:
    dupe_tickers = full_panel[full_panel.duplicated(subset=['ticker','Date'], keep=False)]['ticker'].unique()
    print(f"Affected tickers (investigate before proceeding): {dupe_tickers}")

n_nonpositive_close = (full_panel['Close'] <= 0).sum()
n_negative_volume = (full_panel['Volume'] < 0).sum()
print(f"Non-positive Close values: {n_nonpositive_close}")
print(f"Negative Volume values: {n_negative_volume}")

n_unique_tickers = full_panel['ticker'].nunique()
print(f"\nTotal unique tickers in consolidated panel: {n_unique_tickers} (expect 1070)")

assert dupes == 0, "STOP: duplicate (ticker, Date) rows found -- do not proceed until resolved"
assert n_nonpositive_close == 0, "STOP: non-positive Close values found"
assert n_negative_volume == 0, "STOP: negative Volume values found"
print("\n[PASS] Consolidated panel is clean: no duplicates, no invalid values.")

# ------------------------------------------------------------
# Save, timestamped and versioned -- never overwritten, consistent with
# every other raw file in this project.
# ------------------------------------------------------------
today_str = datetime.now().strftime('%Y-%m-%d')
out_path = BASE + f'raw/prices/raw_price_panel_consolidated_v1_{today_str}.parquet'
full_panel.to_parquet(out_path, index=False)
print(f"\nSaved consolidated raw price panel to: {out_path}")
print(f"This is the FINAL raw price panel for Stage 3 -- ready for the daily")
print(f"PIT-flag join and feature computation in the next stage.")

Found 1070 per-ticker batch files to consolidate.

Consolidated panel shape: (3333105, 9)
Source breakdown:
source
bloomberg    115
yfinance     955
Name: ticker, dtype: int64

FINAL QA ON CONSOLIDATED PANEL

Duplicate (ticker, Date) rows across the WHOLE consolidated panel: 0
Non-positive Close values: 0
Negative Volume values: 0

Total unique tickers in consolidated panel: 1070 (expect 1070)

[PASS] Consolidated panel is clean: no duplicates, no invalid values.

Saved consolidated raw price panel to: /content/drive/MyDrive/moonshot_compounding/raw/prices/raw_price_panel_consolidated_v1_2026-07-14.parquet
This is the FINAL raw price panel for Stage 3 -- ready for the daily
PIT-flag join and feature computation in the next stage.
